# MoveSense · Large-Scale JackRabbot Dataset (JRDB) Benchmark on Google Cloud

This notebook executes the **MoveSense (Topic 06: Pedestrian Tracking and Behaviour Classification)** pipeline on Google Cloud infrastructure utilizing hardware accelerators (NVIDIA T4/A100 GPU) to evaluate the **Stanford JackRabbot Dataset (JRDB-Act)** without taxing local workstation resources.

### Cloud Architecture Highlights:
1. **High-Speed Ingestion:** Data center to data center downloads (multi-gigabit throughput).
2. **GPU Acceleration:** CUDA-accelerated YOLOv8 detector and real-time kinematic extraction.
3. **Academic Evaluation:** Automated computation of Precision, Recall, and F1-Score across all social robotics action classes.

In [ ]:
# 1. Hardware and NVIDIA GPU Verification
!nvidia-smi

In [ ]:
# 2. Clone MoveSense repository and install dependencies
!git clone https://github.com/iancalixto/movesense-project.git || echo 'Repository already exists'
%cd movesense-project
!pip install -q -e .

### 3. Stanford JackRabbot Dataset (JRDB 33.8 GB) Cloud Download
Download the full dataset directly from the Stanford/Monash server into the Google Cloud VM storage.

In [ ]:
# 3. Download the full 33.8 GB Stanford JRDB Dataset directly into Google Cloud VM
!mkdir -p data/JRDB_full

# Direct server-to-server download (takes ~2-3 minutes on Colab)
!wget -c --progress=bar:force:noscroll -O data/train_dataset_with_activity.zip \
    "https://jrdb.erc.monash.edu/static/downloads/JRDB2024/train_dataset_with_activity.zip"

# Unzip and clean archive
!unzip -q data/train_dataset_with_activity.zip -d data/JRDB_full/
!rm data/train_dataset_with_activity.zip

# Verify downloaded sequences
!ls -lh data/JRDB_full/

In [ ]:
# 4. Run the Official MoveSense JRDB Action Benchmark (DS-10)
!python scripts/evaluate_jrdb_benchmark.py

In [ ]:
# 5. Visualize Results & Export Scientific Metrics for Thesis / Presentation
import pandas as pd
import matplotlib.pyplot as plt

df_res = pd.read_csv('results/jrdb_action_benchmark.csv')
display(df_res)

# Action distribution chart
plt.figure(figsize=(9, 4.5))
ax = df_res['Predicao MoveSense'].value_counts().plot(
    kind='bar', color='#1f77b4', edgecolor='black', alpha=0.85
)
plt.title('Predicted Behaviour Distribution on Stanford JRDB-Act Benchmark', fontsize=12, fontweight='bold')
plt.xlabel('Behavioural Class', fontsize=10)
plt.ylabel('Pedestrian Track Count', fontsize=10)
plt.xticks(rotation=30, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('results/jrdb_cloud_distribution.png', dpi=300)
plt.show()